# Is the prior effect a colour artifact?

The prior is signalled by red versus green, and which colour carried the Positive Prior was
counterbalanced. Two questions follow.

1. **Did counterbalancing actually balance?** With 11 and 10 patients, assignment can be uneven by
   chance, and unevenness that lines up with medication or subtype would be a confound.
2. **Does the effect depend on the assignment?** If the medication effect on prior use is really a
   colour effect, it should differ depending on which colour carried the prior. That is the
   `colour assignment x medication` interaction, and its subtype version.

Colour assignment is coded red = +0.5, green = -0.5, and prior direction (which orientation the prior
favoured) left = +0.5, right = -0.5. Both vary within participant, because every patient had a
different mapping in their second session, so their interactions with medication are estimable.

Outputs go to the gitignored `outputs/` directory.


In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2
from imports import *


In [2]:
import pymc as pm

from config import dir_config
from src.utils import classification_utils
from src.utils.mcmc_cache import FIT_SETTINGS, TERMS, prepare_trials, rhat

processed_dir = Path(dir_config.data.processed)
filtered_data = pd.read_csv(processed_dir / "processed_all_data_accu_60_filtered.csv", index_col=None)
metadata = pd.read_csv(processed_dir / "processed_metadata_all_data_accu_60.csv", index_col=None)
subjects_map, _ = classification_utils.get_subject_classification_ids(metadata)
OUTPUT_DIR = (Path.cwd().parent / "outputs").resolve(); OUTPUT_DIR.mkdir(exist_ok=True)

SUBTYPES = {"tremor": "tremor_dominant", "bradykinetic": "bradykinesia_dominant"}
session_info = (metadata.dropna(subset=["treatment"])
                .assign(medication=lambda x: x["treatment"].str.lower())
                .set_index(["subject_id", "medication"]))


PD subjects with ON+OFF sessions: 41
PD subjects with UPDRS subtype: 22
All PD subjects:           41
  Tremor dominant:         11 	 ['P3' 'P6' 'P7' 'P11' 'P12' 'P17' 'P18' 'P19' 'P29' 'P31' 'P32']
  Brady dominant:          10 	 ['P1' 'P4' 'P9' 'P13' 'P20' 'P22' 'P23' 'P28' 'P33' 'P34']
  Intermediate:            1 	 ['P24']
HC subjects:               18 	 ['HC1' 'HC3' 'HC6' 'HC7' 'HC8' 'HC9' 'HC12' 'HC13' 'AV' 'BC' 'BF' 'EM'
 'ES' 'GF' 'GP' 'JA' 'MRM' 'SY']


## Part 1. Did counterbalancing balance?


In [3]:
frames = []
for subtype, key in SUBTYPES.items():
    block = prepare_trials(filtered_data, subjects_map[key]).assign(subtype=subtype)
    info = session_info.loc[list(zip(block["subject_id"], block["medication"]))]
    condition = info["prior_condition"].to_numpy().astype(str)
    block["prior_colour"] = np.where(np.char.startswith(condition, "r"), "red", "green")
    block["prior_direction"] = np.where(np.char.endswith(condition, "l"), "left", "right")
    frames.append(block)
trials = pd.concat(frames, ignore_index=True)
sessions = trials.drop_duplicates(["subject_id", "medication"])[
    ["subject_id", "subtype", "medication", "prior_colour", "prior_direction"]]

for factor in ["prior_colour", "prior_direction"]:
    for grouping in ["medication", "subtype"]:
        table = pd.crosstab(sessions[grouping], sessions[factor])
        odds, p = stats.fisher_exact(table.to_numpy()) if table.shape == (2, 2) else (np.nan, np.nan)
        print(f"\n{factor} x {grouping}  (Fisher exact p = {p:.3f})")
        print(table.to_string())

# does a participant keep the same prior colour across their two sessions?
same = sessions.groupby("subject_id")["prior_colour"].nunique().eq(1)
print(f"\npatients with the same prior colour in both sessions: {same.sum()} of {len(same)}")
same_dir = sessions.groupby("subject_id")["prior_direction"].nunique().eq(1)
print(f"patients with the same prior direction in both sessions: {same_dir.sum()} of {len(same_dir)}")



prior_colour x medication  (Fisher exact p = 0.758)
prior_colour  green  red
medication              
off               9   12
on               11   10

prior_colour x subtype  (Fisher exact p = 0.374)
prior_colour  green  red
subtype                 
bradykinetic      8   12
tremor           12   10

prior_direction x medication  (Fisher exact p = 1.000)
prior_direction  left  right
medication                  
off                 9     12
on                 10     11

prior_direction x subtype  (Fisher exact p = 1.000)
prior_direction  left  right
subtype                     
bradykinetic        9     11
tremor             10     12

patients with the same prior colour in both sessions: 5 of 21
patients with the same prior direction in both sessions: 14 of 21


## Part 2. Does the effect depend on the assignment?

Model 1 adds colour assignment and all its interactions with colour and medication, fitted to both
subtypes' patients. The term of interest is `assign:col:med`: does the medication effect on prior use
differ depending on which physical colour carried the prior?

Model 2 adds subtype, giving `sub:assign:col`, the subtype version of the same question.

Model 3 repeats model 1 for prior direction instead of colour.


In [4]:
def build(block, assign_name, with_subtype=False):
    base = np.column_stack([np.ones(len(block)), block["coh"], block["col"], block["med"],
                            block["col"] * block["med"], block["prev_rc"], block["prev_rc"] * block["med"]])
    names = list(TERMS)
    a = block[assign_name].to_numpy()
    col, med = block["col"].to_numpy(), block["med"].to_numpy()
    extra = [a, a * col, a * med, a * col * med]
    names += ["assign", "assign:col", "assign:med", "assign:col:med"]
    if with_subtype:
        s = block["sub"].to_numpy()
        extra += [s, s * col, s * med, s * col * med, s * a, s * a * col]
        names += ["sub", "sub:col", "sub:med", "sub:col:med", "sub:assign", "sub:assign:col"]
    return np.column_stack([base] + [np.asarray(e)[:, None] for e in extra]).astype("float64"), names


def fit(block, design, seed=0):
    participants = block["subject_id"].unique()
    index = block["subject_id"].map({s: i for i, s in enumerate(participants)}).to_numpy()
    n_re = len(TERMS)
    with pm.Model():
        population = pm.Normal("beta", 0, 2.5, shape=design.shape[1])
        scale = pm.HalfNormal("sd_u", 1.0, shape=n_re)
        z = pm.Normal("z_u", 0, 1, shape=(len(participants), n_re))
        eta = pm.math.sum(design * population, axis=1) + pm.math.sum(design[:, :n_re] * (z * scale)[index], axis=1)
        pm.Bernoulli("obs", logit_p=eta, observed=block["choice"].to_numpy().astype("int8"))
        return pm.sample(FIT_SETTINGS["draws"], tune=FIT_SETTINGS["tune"], chains=FIT_SETTINGS["chains"],
                         cores=FIT_SETTINGS["chains"], target_accept=0.95,
                         random_seed=seed, progressbar=False)


trials["red"] = np.where(trials["prior_colour"] == "red", 0.5, -0.5)
trials["left"] = np.where(trials["prior_direction"] == "left", 0.5, -0.5)
trials["sub"] = np.where(trials["subtype"] == "tremor", 0.5, -0.5)

MODELS = [("colour assignment", "red", False), ("colour assignment x subtype", "red", True),
          ("prior direction", "left", False)]

results, posteriors = [], {}
for label, assign, with_subtype in MODELS:
    design, names = build(trials, assign, with_subtype)
    idata = fit(trials, design)
    beta = idata.posterior["beta"].values
    posteriors[label] = (beta, names)
    report = ["col:med", "assign:col", "assign:col:med"] + (["sub:col:med", "sub:assign:col"] if with_subtype else [])
    for term in report:
        draws = beta[..., names.index(term)].ravel()
        low, high = np.percentile(draws, [2.5, 97.5])
        results.append({"model": label, "term": term, "mean": draws.mean(), "2.5%": low, "97.5%": high,
                        "P(> 0)": (draws > 0).mean(),
                        "divergences": int(idata.sample_stats["diverging"].values.sum()),
                        "max r_hat": max(rhat(beta[..., k]) for k in range(beta.shape[-1]))})
        print(f"{label:28s} {term:16s} {draws.mean():+.3f} [{low:+.3f}, {high:+.3f}] P(>0)={(draws > 0).mean():.3f}")

artifact = pd.DataFrame(results).set_index(["model", "term"])
artifact.round(3)


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, sd_u, z_u]
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 194 seconds.
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...


colour assignment            col:med          +0.252 [-0.097, +0.582] P(>0)=0.928
colour assignment            assign:col       -0.473 [-0.812, -0.125] P(>0)=0.004
colour assignment            assign:col:med   -0.088 [-0.744, +0.554] P(>0)=0.399


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, sd_u, z_u]
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 337 seconds.
Initializing NUTS using jitter+adapt_diag...


colour assignment x subtype  col:med          +0.257 [-0.080, +0.608] P(>0)=0.936
colour assignment x subtype  assign:col       -0.445 [-0.794, -0.098] P(>0)=0.006
colour assignment x subtype  assign:col:med   -0.024 [-0.714, +0.650] P(>0)=0.484
colour assignment x subtype  sub:col:med      +0.400 [-0.289, +1.099] P(>0)=0.878
colour assignment x subtype  sub:assign:col   +0.007 [-0.678, +0.680] P(>0)=0.508


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, sd_u, z_u]
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 246 seconds.


prior direction              col:med          +0.309 [-0.043, +0.684] P(>0)=0.955
prior direction              assign:col       +0.018 [-0.335, +0.370] P(>0)=0.546
prior direction              assign:col:med   +0.196 [-0.560, +0.934] P(>0)=0.699


mean   2.5%  97.5%  P(> 0)  \
model                       term                                          
colour assignment           col:med         0.252 -0.097  0.582   0.928   
                            assign:col     -0.473 -0.812 -0.125   0.004   
                            assign:col:med -0.088 -0.744  0.554   0.399   
colour assignment x subtype col:med         0.257 -0.080  0.608   0.936   
                            assign:col     -0.445 -0.794 -0.098   0.006   
                            assign:col:med -0.024 -0.714  0.650   0.484   
                            sub:col:med     0.400 -0.289  1.099   0.878   
                            sub:assign:col  0.007 -0.678  0.680   0.508   
prior direction             col:med         0.309 -0.043  0.684   0.955   
                            assign:col      0.018 -0.335  0.370   0.546   
                            assign:col:med  0.196 -0.560  0.934   0.698   

                                            divergences  max r_hat  
model                       term                                    
colour assignment           col:med                   0      1.008  
                            assign:col                0      1.008  
                            assign:col:med            0      1.008  
colour assignment x subtype col:med                   0      1.002  
                            assign:col                0      1.002  
                            assign:col:med            0      1.002  
                            sub:col:med               0      1.002  
                            sub:assign:col            0      1.002  
prior direction             col:med                   0      1.007  
                            assign:col                0      1.007  
                            assign:col:med            0      1.007

In [5]:
artifact.to_csv(OUTPUT_DIR / "colour_artifact_tests.csv")
sessions.to_csv(OUTPUT_DIR / "colour_assignment_balance.csv", index=False)
print("saved colour-artifact tests and the assignment table")


saved colour-artifact tests and the assignment table
